<a href="https://colab.research.google.com/github/nishthadighe-bit/Data--Engineering-Practicals/blob/main/Practical_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import json
import xml.etree.ElementTree as ET
import numpy as np

# 1. Parsing CSV
csv_data = """Name,Age,Salary
Amit,25,50000
Riya,,60000
Neha,28,
Unknown,30,55000"""

with open('data.csv', 'w') as f:
    f.write(csv_data)

df_csv = pd.read_csv('data.csv')
print("--- CSV Data & Missing Values ---")
print(df_csv)
print("\nMissing Values Count:\n", df_csv.isnull().sum())

# 2. Parsing JSON
json_data = '''[
    {"id": 1, "product": "Laptop", "price": 1000},
    {"id": 2, "product": "Mouse", "price": null},
    {"id": 3, "product": null, "price": 150}
]'''
df_json = pd.read_json(json_data)
print("\n--- JSON Data ---")
print(df_json)

# 3. Parsing XML
xml_data = """<records>
    <student><name>Aman</name><marks>85</marks></student>
    <student><name>Priya</name><marks></marks></student>
</records>"""
root = ET.fromstring(xml_data)
students = []
for s in root.findall('student'):
    name = s.find('name').text
    marks = s.find('marks').text
    students.append({'Name': name, 'Marks': marks if marks else np.nan})

df_xml = pd.DataFrame(students)
print("\n--- XML Data ---")
print(df_xml)

--- CSV Data & Missing Values ---
      Name   Age   Salary
0     Amit  25.0  50000.0
1     Riya   NaN  60000.0
2     Neha  28.0      NaN
3  Unknown  30.0  55000.0

Missing Values Count:
 Name      0
Age       1
Salary    1
dtype: int64

--- JSON Data ---
   id product   price
0   1  Laptop  1000.0
1   2   Mouse     NaN
2   3    None   150.0

--- XML Data ---
    Name Marks
0   Aman    85
1  Priya   NaN


/tmp/ipykernel_1015/329137343.py:27: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_json = pd.read_json(json_data)


In [ ]:
import pickle

data_dict = {'project': 'Data Handling', 'version': 1.0, 'status': 'Active'}

# Writing Binary (Pickling)
with open('data.bin', 'wb') as bin_file:
    pickle.dump(data_dict, bin_file)
print("Binary file 'data.bin' written successfully.")

# Reading Binary (Unpickling)
with open('data.bin', 'rb') as bin_file:
    loaded_data = pickle.load(bin_file)

print("Read from Binary File:", loaded_data)

Binary file 'data.bin' written successfully.
Read from Binary File: {'project': 'Data Handling', 'version': 1.0, 'status': 'Active'}


In [ ]:
import re

text = "Contact us at support@example.com or admin@domain.org for queries. Call 987-654-3210."

# 1. Search (Extract emails)
emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', text)
print("Extracted Emails:", emails)

# 2. Split (Split text by punctuation/whitespace)
words = re.split(r'\s+|,|\.', text)
print("\nSplit Words:", [w for w in words if w])

# 3. Replace (Mask phone numbers)
masked_text = re.sub(r'\d{3}-\d{3}-\d{4}', 'XXX-XXX-XXXX', text)
print("\nMasked Text:", masked_text)

Extracted Emails: ['support@example.com', 'admin@domain.org']

Split Words: ['Contact', 'us', 'at', 'support@example', 'com', 'or', 'admin@domain', 'org', 'for', 'queries', 'Call', '987-654-3210']

Masked Text: Contact us at support@example.com or admin@domain.org for queries. Call XXX-XXX-XXXX.


In [ ]:
import sqlite3
import pandas as pd

# 1. Design & Connect to Relational Database
conn = sqlite3.connect('student_db.db')
cursor = conn.cursor()

cursor.execute('DROP TABLE IF EXISTS students')
cursor.execute('''
CREATE TABLE students (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER,
    grade TEXT
)
''')

# 2. Insert Data (Create)
cursor.executemany('''
INSERT INTO students (name, age, grade) VALUES (?, ?, ?)
''', [('Amit', 21, 'A'), ('Riya', 22, 'B'), ('Pooja', 20, 'A')])
conn.commit()

# 3. Read Data
print("--- Initial Database State ---")
print(pd.read_sql_query("SELECT * FROM students", conn))

# 4. Update Data
cursor.execute("UPDATE students SET grade = 'A+' WHERE name = 'Riya'")
conn.commit()

# 5. Delete Data
cursor.execute("DELETE FROM students WHERE name = 'Pooja'")
conn.commit()

# 6. Final Read Verification
print("\n--- Final Database State (After Update & Delete) ---")
print(pd.read_sql_query("SELECT * FROM students", conn))

conn.close()

--- Initial Database State ---
   id   name  age grade
0   1   Amit   21     A
1   2   Riya   22     B
2   3  Pooja   20     A

--- Final Database State (After Update & Delete) ---
   id  name  age grade
0   1  Amit   21     A
1   2  Riya   22    A+
